# FREUID Challenge 2026 (IJCAI-ECAI) — Production Notebook v3 - Diagnostics

## Fold zero checkpoint saved as dataset and added as input

**Task:** Binary fraud detection on identity document images  
**Metric:** FREUID Score (lower is better) = 1 minus harmonic-mean(g_audet, g_apcer)  
**Data:** `/kaggle/input/datasets/maheshwarmishra/freuid-data/`  
**Weights:** `/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b4/1/`  
**Output:** `/kaggle/working/`

---
## Structure (all definitions before execution)
1. Environment Setup  
2. Config  
3. Data Loading and EDA  
4. FREUID Metric  
5. Cross-Validation Splits  
6. Augmentation Pipeline  
7. Dataset Class and Weighted Sampler  
8. Model Architecture  
9. Loss Functions  
10. Score Calibration  
11. Inference and Submission  
12. Training Pipeline  
13. Ensemble Utilities  
14. Visualization and Ablation Logger  
15. Quick-Start Reference Cells  
16. EXECUTION (runs after all definitions)  

## 1. Environment Setup

In [ ]:
import os
import time
import math
import json
import random
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

# Must be set BEFORE importing albumentations and timm.
# Kaggle sandboxed network blocks outbound DNS to huggingface.co and PyPI.
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['HF_HUB_OFFLINE']           = '1'
os.environ['TRANSFORMERS_OFFLINE']     = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import scipy.optimize
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy  : {np.__version__}')
print(f'timm   : {timm.__version__}')

## 2. Config

In [ ]:
@dataclass
class CFG:
    # Paths
    # Dataset: /kaggle/input/datasets/maheshwarmishra/freuid-data/
    # Structure:
    #   /kaggle/input/datasets/maheshwarmishra/freuid-data/train/           (training images)
    #   /kaggle/input/datasets/maheshwarmishra/freuid-data/public_test/     (7821 test images)
    #   /kaggle/input/datasets/maheshwarmishra/freuid-data/train_labels.csv
    #   /kaggle/input/datasets/maheshwarmishra/freuid-data/sample_submission.csv (142818 rows, column=label)
    # Weights: /kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b4/1/
    # Private test (134997 images) not yet released. Dummy scores (fraud_rate) fill those IDs.
    # Host confirmed dummy scores are ignored on public leaderboard (competition discussion).
    # Private test includes 2 document types UNSEEN in train or public_test.
    DATA_DIR:          str   = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
    WEIGHTS_PATH:      str   = ('/kaggle/input/models/timm/tf-efficientnet/'
                                'pytorch/tf-efficientnet-b4/1/'
                                'tf_efficientnet_b4_aa-818f208c.pth')
    OUTPUT_DIR:        str   = '/kaggle/working'
    CHECKPOINT_DIR:    str   = '/kaggle/working/checkpoints'
    USE_FULL_DATA:     bool  = True
    TRAIN_LABELS_FILE: str   = ''
    TRAIN_IMG_SUBDIR:  str   = ''
    TEST_IMG_SUBDIR:   str   = 'public_test'

    # Model
    # tf_efficientnet_b4 matches the Kaggle-hosted weights file exactly.
    # Apache-2.0 via timm (OSI-compatible, prize-eligible).
    BACKBONE:          str   = 'tf_efficientnet_b4'
    PRETRAINED:        bool  = True
    IMG_SIZE:          int   = 320
    DROP_RATE:         float = 0.3
    USE_METADATA:      bool  = True
    N_DOC_TYPES:       int   = 256
    DOC_EMB_DIM:       int   = 16

    # Training
    # GPU memory note: IMG_SIZE=320 + EfficientNet-B4 + AMP=True uses ~10-12GB
    # on a T4 (16GB). If OOM: reduce BATCH_SIZE to 8, increase GRAD_ACCUM to 4
    # to keep effective batch size = 32.
    N_FOLDS:           int   = 5
    EPOCHS:            int   = 20
    BATCH_SIZE:        int   = 16
    GRAD_ACCUM:        int   = 2
    NUM_WORKERS:       int   = 0
    PIN_MEMORY:        bool  = True
    SAMPLER_CAP:       int   = -1
    VAL_SAMPLE_SIZE:   int   = 5000

    # Optimizer
    LR:                float = 2e-4
    LR_MIN:            float = 1e-6
    WEIGHT_DECAY:      float = 1e-4
    WARMUP_EPOCHS:     int   = 2
    GRAD_CLIP:         float = 1.0

    # Loss: pauc_focal directly optimizes the 1% BPCER operating region
    LOSS_TYPE:         str   = 'pauc_focal'
    FOCAL_GAMMA:       float = 2.0
    FOCAL_ALPHA:       float = 0.75
    FPR_MAX:           float = 0.10
    PAUC_WEIGHT:       float = 0.5

    # Augmentation (full probabilities for production run)
    AUG_P_JPEG:        float = 0.5
    AUG_P_NOISE:       float = 0.3
    AUG_P_BLUR:        float = 0.3
    AUG_P_MOIRE:       float = 0.2
    AUG_P_WARP:        float = 0.15

    # Calibration
    CALIBRATE:         bool  = True
    CALIB_METHOD:      str   = 'temperature'

    # Mixed precision
    AMP:               bool  = True

    # W&B
    USE_WANDB:         bool  = False
    WANDB_PROJECT:     str   = 'freuid-2026'
    WANDB_RUN_NAME:    str   = 'effb4-pauc-production'

    # Evaluation
    BPCER_TARGET:      float = 0.01

    def __post_init__(self):
        if not self.TRAIN_LABELS_FILE:
            self.TRAIN_LABELS_FILE = (
                'train_labels.csv' if self.USE_FULL_DATA
                else 'train_sample_labels.csv'
            )
        if not self.TRAIN_IMG_SUBDIR:
            self.TRAIN_IMG_SUBDIR = 'train' if self.USE_FULL_DATA else 'train_sample'


CFG = CFG()
Path(CFG.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded.')
print(f'  DATA_DIR         : {CFG.DATA_DIR}')
print(f'  TRAIN_LABELS_FILE: {CFG.TRAIN_LABELS_FILE}')
print(f'  BACKBONE         : {CFG.BACKBONE}')
print(f'  IMG_SIZE         : {CFG.IMG_SIZE}')
print(f'  LOSS_TYPE        : {CFG.LOSS_TYPE}')

## 3. Data Loading and EDA

In [ ]:
IMAGE_EXTENSIONS = ('.jpeg', '.jpg', '.png', '.webp', '.bmp')

def find_images_in_dir(directory: Path) -> List[Path]:
    files = []
    for ext in IMAGE_EXTENSIONS:
        files += list(directory.glob(f'*{ext}'))
    return sorted(files)


def find_images_robust(base_dir: Path, subdir: str) -> Tuple[Path, List[Path]]:
    """
    Find images under base_dir/subdir handling any single level of nesting.
    Handles the doubled-folder quirk from Kaggle zip uploads:
      /kaggle/input/datasets/maheshwarmishra/freuid-data/public_test/public_test/<uuid>.jpeg
    Returns (actual_image_dir, sorted_image_paths).
    """
    flat = base_dir / subdir
    files = find_images_in_dir(flat) if flat.exists() else []
    if files:
        return flat, files
    doubled = base_dir / subdir / subdir
    files = find_images_in_dir(doubled) if doubled.exists() else []
    if files:
        return doubled, files
    if flat.exists():
        for child in sorted(flat.iterdir()):
            if child.is_dir():
                files = find_images_in_dir(child)
                if files:
                    return child, files
    return flat, []


data_dir = Path(CFG.DATA_DIR)

# Train labels
# File: /kaggle/input/datasets/maheshwarmishra/freuid-data/train_labels.csv
labels_path = data_dir / CFG.TRAIN_LABELS_FILE
if not labels_path.exists():
    raise FileNotFoundError(
        f'Labels file not found: {labels_path}\n'
        f'Check CFG.DATA_DIR={CFG.DATA_DIR}'
    )
train_df = pd.read_csv(labels_path)
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

required = {'id', 'image_path', 'label', 'is_digital', 'type'}
missing  = required - set(train_df.columns)
if missing:
    raise ValueError(f'train_df missing columns: {missing}')

# Test set
# Location: /kaggle/input/datasets/maheshwarmishra/freuid-data/public_test/
# 7821 public test images. 134997 private test images not yet released.
actual_test_dir, test_files = find_images_robust(data_dir, CFG.TEST_IMG_SUBDIR)
if test_files:
    rel_prefix = actual_test_dir.relative_to(data_dir)
    test_df = pd.DataFrame({
        'id':         [p.stem for p in test_files],
        'image_path': [str(rel_prefix / p.name) for p in test_files],
        'is_digital': 0,
        'type_idx':   0,
    })
    print(f'Test images found: {len(test_df)} in {actual_test_dir}')
else:
    print(f'WARNING: No test images found under {data_dir / CFG.TEST_IMG_SUBDIR}')
    test_df = pd.DataFrame(columns=['id', 'image_path', 'is_digital', 'type_idx'])

# Sample submission
# File: /kaggle/input/datasets/maheshwarmishra/freuid-data/sample_submission.csv
# 142818 rows total (7821 public + 134997 private). Score column is named 'label'.
sub_csv = data_dir / 'sample_submission.csv'
sub_df  = pd.read_csv(sub_csv) if sub_csv.exists() else pd.DataFrame()
if not sub_df.empty:
    print(f'Sample submission: {len(sub_df)} rows, columns: {sub_df.columns.tolist()}')

print(f'Train shape: {train_df.shape}')
print(f'Fraud rate : {train_df["label"].mean():.3f}')
train_df.head()

In [ ]:
# EDA
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
train_df['label'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#2196F3', '#F44336'])
axes[0].set_title('Label Distribution')
axes[0].set_xticklabels(['Genuine', 'Fraud'], rotation=0)

pd.crosstab(train_df['is_digital'], train_df['label']).plot(
    kind='bar', ax=axes[1], color=['#2196F3', '#F44336'])
axes[1].set_title('is_digital x label')
axes[1].set_xticklabels(['Recaptured', 'Digital'], rotation=0)
axes[1].legend(['Genuine', 'Fraud'])

train_df['type'].value_counts().head(15).plot(
    kind='barh', ax=axes[2], color='#9C27B0')
axes[2].set_title('Top-15 Document Types')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Unique document types: {train_df["type"].nunique()}')

In [ ]:
# Document type to integer index
# Vocab built from train only. Public test has no type column.
# Private test has 2 UNSEEN types -> index 0 (<UNK>) at inference.
all_types = train_df['type'].unique()
type2idx  = {t: i + 1 for i, t in enumerate(sorted(all_types))}
type2idx['<UNK>'] = 0
train_df['type_idx'] = train_df['type'].map(type2idx).fillna(0).astype(int)
if 'type_idx' not in test_df.columns:
    test_df['type_idx'] = 0
CFG.N_DOC_TYPES = len(type2idx) + 1
print(f'Document type vocabulary size: {CFG.N_DOC_TYPES}')

In [ ]:
# Standalone image path resolver (used in EDA before FREUIDDataset is defined)
# FREUIDDataset has its own internal _resolve method with identical logic.
def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    c = base_dir / rel_path
    if c.exists(): return c
    parts = Path(rel_path).parts
    if len(parts) > 1:
        d = base_dir / parts[0] / rel_path
        if d.exists(): return d
    f = base_dir / Path(rel_path).name
    if f.exists(): return f
    return c


def show_samples(df: pd.DataFrame, n: int = 8, title: str = '') -> None:
    sample = df.sample(n=min(n, len(df)), random_state=SEED)
    cols   = min(n, 4)
    rows   = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 5 * rows))
    for ax, (_, row) in zip(np.array(axes).flat, sample.iterrows()):
        img_path = resolve_image_path(data_dir, row['image_path'])
        try:
            ax.imshow(Image.open(img_path).convert('RGB'))
        except Exception:
            ax.text(0.5, 0.5, 'Load error', ha='center', va='center',
                    transform=ax.transAxes)
        lbl = 'FRAUD'   if row.get('label',      -1) == 1 else 'GENUINE'
        dig = 'digital' if row.get('is_digital',  -1) == 1 else 'recaptured'
        ax.set_title(f"{lbl}\n{row.get('type','?')} | {dig}", fontsize=8)
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_samples(train_df, n=8, title='Random Training Samples')

## 4. FREUID Metric

In [ ]:
def compute_det_curve(
    labels: np.ndarray, scores: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    thresholds = np.unique(scores)[::-1]
    n_genuine  = (labels == 0).sum()
    n_attack   = (labels == 1).sum()
    assert n_genuine > 0 and n_attack > 0, (
        f'Need both classes. Got {n_genuine} genuine, {n_attack} attack.')
    far_list, frr_list = [], []
    for thresh in thresholds:
        preds = (scores >= thresh).astype(int)
        FP = ((preds == 1) & (labels == 0)).sum()
        FN = ((preds == 0) & (labels == 1)).sum()
        TP = ((preds == 1) & (labels == 1)).sum()
        TN = ((preds == 0) & (labels == 0)).sum()
        far_list.append(FP / (FP + TN) if (FP + TN) > 0 else 0.0)
        frr_list.append(FN / (FN + TP) if (FN + TP) > 0 else 0.0)
    return thresholds, np.array(far_list), np.array(frr_list)


def compute_audet(far: np.ndarray, frr: np.ndarray) -> float:
    order = np.argsort(far)
    far_s = np.concatenate([[0.0], far[order], [1.0]])
    frr_s = np.concatenate([[1.0], frr[order], [0.0]])
    trapz = getattr(np, 'trapezoid', None) or np.trapz
    return float(trapz(frr_s, far_s))


def compute_apcer_at_bpcer(
    labels: np.ndarray, scores: np.ndarray, bpcer_target: float = 0.01
) -> float:
    _, far, frr = compute_det_curve(labels, scores)
    valid = far <= bpcer_target
    return float(frr[valid].min()) if valid.any() else 1.0


def compute_freuid_score(
    labels: np.ndarray, scores: np.ndarray,
    bpcer_target: float = 0.01, eps: float = 1e-8
) -> Dict[str, float]:
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    _, far, frr = compute_det_curve(labels, scores)
    audet      = compute_audet(far, frr)
    apcer_1pct = compute_apcer_at_bpcer(labels, scores, bpcer_target)
    g_audet = 1.0 - audet
    g_apcer = 1.0 - apcer_1pct
    denom   = g_audet + g_apcer
    freuid  = (1.0 - (2.0 * g_audet * g_apcer) / denom
               if abs(denom) > eps else 1.0)
    try:
        auc_roc = roc_auc_score(labels, scores)
    except Exception:
        auc_roc = float('nan')
    return {'freuid': freuid, 'audet': audet, 'apcer_1pct': apcer_1pct,
            'g_audet': g_audet, 'g_apcer': g_apcer, 'auc_roc': auc_roc}


def evaluate_breakdown(
    df: pd.DataFrame, scores: np.ndarray, bpcer_target: float = 0.01
) -> pd.DataFrame:
    # reset_index ensures df.index is a clean RangeIndex (0,1,2,...)
    # so that grp.index values are positional and scores[grp.index] is safe.
    # Fix: scores is a numpy array indexed by position, not by DataFrame index.
    # Without reset_index, grp.index would reference original df index values
    # which may not align with numpy array positions.
    df    = df.reset_index(drop=True)
    rows  = []
    m = compute_freuid_score(df['label'].values, scores, bpcer_target)
    rows.append({'group': 'OVERALL', 'subset': 'all', 'n': len(df), **m})
    for dig_val, dig_name in [(0, 'recaptured'), (1, 'digital')]:
        mask = (df['is_digital'] == dig_val).values
        if mask.sum() < 10 or len(np.unique(df['label'].values[mask])) < 2:
            continue
        m = compute_freuid_score(
            df['label'].values[mask], scores[mask], bpcer_target)
        rows.append({'group': 'is_digital', 'subset': dig_name,
                     'n': int(mask.sum()), **m})
    for doc_type, grp in df.groupby('type'):
        if len(grp) < 20 or len(grp['label'].unique()) < 2:
            continue
        # grp.index is a RangeIndex subset after reset_index above,
        # so these values are valid positional indices into scores array.
        m = compute_freuid_score(
            grp['label'].values, scores[grp.index.values], bpcer_target)
        rows.append({'group': 'doc_type', 'subset': doc_type,
                     'n': len(grp), **m})
    return pd.DataFrame(rows)


def plot_det_curve(
    labels: np.ndarray, scores: np.ndarray, title: str = 'DET_Curve'
) -> None:
    _, far, frr = compute_det_curve(labels, scores)
    audet = compute_audet(far, frr)
    apcer = compute_apcer_at_bpcer(labels, scores, CFG.BPCER_TARGET)
    order = np.argsort(far)
    plt.figure(figsize=(6, 6))
    plt.plot(far[order], frr[order], 'b-', lw=2, label=f'AuDET={audet:.4f}')
    plt.axvline(CFG.BPCER_TARGET, color='red', ls='--',
                label=f'BPCER={CFG.BPCER_TARGET*100:.0f}%')
    plt.scatter([CFG.BPCER_TARGET], [apcer], color='red', zorder=5,
                label=f'APCER@1%={apcer:.4f}')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    plt.fill_between(far[order], frr[order], alpha=0.08)
    plt.xlabel('FAR (BPCER)')
    plt.ylabel('FRR (APCER)')
    plt.title(title)
    plt.legend()
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.tight_layout()
    safe = title.replace(' ', '_').replace('/', '_')
    plt.savefig(f'{CFG.OUTPUT_DIR}/det_{safe}.png', dpi=120, bbox_inches='tight')
    plt.show()


# Sanity checks
np.random.seed(0)
_lbl = np.random.binomial(1, 0.5, 500)
_rnd = np.random.uniform(0, 1, 500)
_prf = _lbl.astype(float) + np.random.uniform(0, 0.01, 500)
m_rnd = compute_freuid_score(_lbl, _rnd)
m_prf = compute_freuid_score(_lbl, _prf)
print(f'Random  : FREUID={m_rnd["freuid"]:.4f} AUC={m_rnd["auc_roc"]:.4f} (expect ~1.0, ~0.5)')
print(f'Perfect : FREUID={m_prf["freuid"]:.4f} AUC={m_prf["auc_roc"]:.4f} (expect ~0.0, ~1.0)')
print('FREUID metric: OK')

## 5. Cross-Validation Splits

In [ ]:
train_df['strat_key'] = (
    train_df['label'].astype(str) + '_' +
    train_df['is_digital'].astype(str) + '_' +
    train_df['type'].astype(str)
)
min_stratum  = train_df['strat_key'].value_counts().min()
n_folds_safe = max(2, min(CFG.N_FOLDS, min_stratum))
if n_folds_safe < CFG.N_FOLDS:
    print(f'Smallest stratum={min_stratum} -> using n_splits={n_folds_safe}')

skf = StratifiedKFold(n_splits=n_folds_safe, shuffle=True, random_state=SEED)
train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(
        skf.split(train_df, train_df['strat_key'])):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold_idx

assert (train_df['fold'] == -1).sum() == 0, 'Some rows not assigned a fold!'
print(f'Folds created: {n_folds_safe}')
print(train_df.groupby('fold')['label'].value_counts().unstack())

## 6. Augmentation Pipeline

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def build_recapture_aug() -> A.Compose:
    """
    Physics-inspired augmentations simulating the print-and-capture analog hole.
    Teaches the model that physical capture degradations alone do not equal fraud.
    Critical for generalization to captured/recaptured test examples.
    Full probabilities for production run.
    """
    return A.Compose([
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Defocus(radius=(1, 3), p=1.0),
        ], p=CFG.AUG_P_BLUR),
        A.OneOf([
            A.GaussNoise(var_limit=(5, 30), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
        ], p=CFG.AUG_P_NOISE),
        A.ImageCompression(quality_lower=40, quality_upper=85, p=CFG.AUG_P_JPEG),
        A.Perspective(scale=(0.02, 0.08), p=CFG.AUG_P_WARP),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.GridDistortion(num_steps=4, distort_limit=0.08, p=CFG.AUG_P_MOIRE),
    ], p=1.0)


def build_train_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=10, border_mode=0, p=0.5),
        A.CoarseDropout(max_holes=4, max_height=img_size // 8,
                        max_width=img_size // 8, fill_value=0, p=0.3),
        build_recapture_aug(),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print(f'Transforms defined. IMG_SIZE={CFG.IMG_SIZE}')

## 7. Dataset Class and Weighted Sampler

In [ ]:
class FREUIDDataset(Dataset):
    """
    Document fraud detection dataset.
    Data directory: /kaggle/input/datasets/maheshwarmishra/freuid-data/
    Handles doubled-folder path quirk from Kaggle zip uploads.
    """

    def __init__(self, df: pd.DataFrame, data_dir: Path,
                 transform: Optional[A.Compose] = None,
                 is_train: bool = True) -> None:
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform
        self.is_train  = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, rel_path: str) -> Path:
        c = self.data_dir / rel_path
        if c.exists(): return c
        parts = Path(rel_path).parts
        if len(parts) > 1:
            d = self.data_dir / parts[0] / rel_path
            if d.exists(): return d
        f = self.data_dir / Path(rel_path).name
        if f.exists(): return f
        return c

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        img_path = self._resolve(row['image_path'])
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Warning: cannot load {img_path}: {e}')
            image = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
        if self.transform is not None:
            image = self.transform(image=image)['image']
        out = {
            'image':      image,
            'is_digital': torch.tensor(float(row.get('is_digital', 0)),
                                       dtype=torch.float32),
            'type_idx':   torch.tensor(int(row.get('type_idx', 0)),
                                       dtype=torch.long),
            'id':         str(row['id']),
        }
        if self.is_train:
            out['label'] = torch.tensor(int(row['label']), dtype=torch.long)
        return out


def build_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """
    Vectorized weighted sampler (no Python loop over rows).
    Balances label classes and up-weights rare (type x is_digital) strata.
    SAMPLER_CAP=-1 uses full dataset length.
    """
    label_w = df['label'].map(
        (1.0 / df['label'].value_counts()).to_dict()
    ).values.astype(float)
    strat_w = (
        1.0 / df.groupby(['type', 'is_digital'])['label']
        .transform('count').values.astype(float)
    )
    weights   = torch.tensor(label_w * strat_w, dtype=torch.double)
    n_samples = (len(weights) if CFG.SAMPLER_CAP <= 0
                 else min(len(weights), CFG.SAMPLER_CAP))
    return WeightedRandomSampler(weights, num_samples=n_samples, replacement=True)


print('FREUIDDataset and build_weighted_sampler defined.')

## 8. Model Architecture

In [ ]:
class FREUIDModel(nn.Module):
    """
    EfficientNet-B4 backbone + metadata conditioning head.
    Weights: /kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b4/1/
    Loaded locally (HF_HUB_OFFLINE=1 blocks all network calls).
    Apache-2.0 via timm (OSI-compatible, prize-eligible).
    """

    def __init__(
        self,
        backbone_name: str   = CFG.BACKBONE,
        pretrained:    bool  = CFG.PRETRAINED,
        weights_path:  str   = CFG.WEIGHTS_PATH,
        n_doc_types:   int   = CFG.N_DOC_TYPES,
        doc_emb_dim:   int   = CFG.DOC_EMB_DIM,
        drop_rate:     float = CFG.DROP_RATE,
        use_metadata:  bool  = CFG.USE_METADATA,
    ) -> None:
        super().__init__()
        self.use_metadata = use_metadata

        self.backbone = timm.create_model(
            backbone_name, pretrained=False, num_classes=0, global_pool='avg')

        if pretrained:
            wp = Path(weights_path)
            if wp.exists():
                state_dict = torch.load(wp, map_location='cpu', weights_only=False)
                missing, unexpected = self.backbone.load_state_dict(
                    state_dict, strict=False)
                print(f'  Backbone weights loaded: {wp.name}')
                if missing:
                    print(f'  Missing keys   : {len(missing)} (expected, head removed)')
                if unexpected:
                    n = len(unexpected)
                    print(f'  Unexpected keys: {n}{" inspect if > 5" if n > 5 else ""}')
            else:
                print(f'  WARNING: weights not found at {weights_path}')
                print('  Training from random init.')

        feat_dim = self.backbone.num_features

        if use_metadata:
            self.doc_embedding = nn.Embedding(
                num_embeddings=n_doc_types, embedding_dim=doc_emb_dim, padding_idx=0)
            meta_dim = doc_emb_dim + 1
        else:
            meta_dim = 0

        in_dim    = feat_dim + meta_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(drop_rate),
            nn.Linear(in_dim, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2), nn.Linear(256, 1),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, image: torch.Tensor,
                is_digital: torch.Tensor,
                type_idx: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(image)
        if self.use_metadata:
            feats = torch.cat(
                [feats, self.doc_embedding(type_idx), is_digital.unsqueeze(1)],
                dim=1)
        return self.head(feats).squeeze(1)


_m = FREUIDModel().to(DEVICE)
total     = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Model: Total={total/1e6:.1f}M Trainable={trainable/1e6:.1f}M')
_img  = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(DEVICE)
_dig  = torch.tensor([0.0, 1.0]).to(DEVICE)
_tidx = torch.tensor([1, 2]).to(DEVICE)
with torch.no_grad():
    _out = _m(_img, _dig, _tidx)
print(f'Forward pass OK: output={_out.shape}, values={_out}')
del _m, _img, _dig, _tidx, _out
torch.cuda.empty_cache()

## 9. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs   = torch.sigmoid(logits)
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


class PartialAUCLoss(nn.Module):
    """
    Optimizes pairwise ranking between attacks and the hardest genuine samples
    (top fpr_max% by score). Directly targets the 1% BPCER operating point.
    """
    def __init__(self, fpr_max=CFG.FPR_MAX, margin=1.0):
        super().__init__()
        self.fpr_max = fpr_max
        self.margin  = margin

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs   = torch.sigmoid(logits)
        targets = targets.float()
        attack  = probs[targets == 1]
        genuine = probs[targets == 0]
        if len(attack) == 0 or len(genuine) == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        k = max(1, int(math.ceil(self.fpr_max * len(genuine))))
        hard_genuine, _ = torch.topk(genuine, k=k)
        diff = attack.unsqueeze(1) - hard_genuine.unsqueeze(0)
        return F.relu(self.margin - diff).mean()


class CombinedLoss(nn.Module):
    def __init__(self, pauc_weight=CFG.PAUC_WEIGHT, fpr_max=CFG.FPR_MAX):
        super().__init__()
        self.focal = FocalLoss()
        self.pauc  = PartialAUCLoss(fpr_max=fpr_max)
        self.w     = pauc_weight

    def forward(self, logits, targets):
        focal_loss = self.focal(logits, targets)
        pauc_loss  = self.pauc(logits, targets)
        total      = (1 - self.w) * focal_loss + self.w * pauc_loss
        return total, {'focal_loss': focal_loss.item(), 'pauc_loss': pauc_loss.item()}


def get_loss_fn() -> nn.Module:
    if CFG.LOSS_TYPE == 'focal':      return FocalLoss()
    if CFG.LOSS_TYPE == 'pauc_focal': return CombinedLoss()
    if CFG.LOSS_TYPE == 'bce':        return nn.BCEWithLogitsLoss()
    raise ValueError(f'Unknown loss type: {CFG.LOSS_TYPE}')


print('FocalLoss, PartialAUCLoss, CombinedLoss, get_loss_fn defined.')

## 10. Score Calibration

In [ ]:
class TemperatureScaler:
    """
    Post-hoc temperature scaling (Guo et al., 2017).
    Learns T>0 s.t. sigmoid(logit/T) is better calibrated.
    Fit on val fold only (never on training data).
    Mandatory: APCER@1%BPCER depends on score distribution near a fixed threshold.
    """
    def __init__(self): self.temperature = 1.0

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'TemperatureScaler':
        def nll(t):
            t = float(t[0])
            if t <= 0: return 1e9
            p = np.clip(1.0 / (1.0 + np.exp(-logits / t)), 1e-7, 1 - 1e-7)
            return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        res = scipy.optimize.minimize(nll, [1.0], method='L-BFGS-B',
                                      bounds=[(0.05, 20.0)])
        self.temperature = float(res.x[0])
        print(f'  Temperature = {self.temperature:.4f}')
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        return (1.0 / (1.0 + np.exp(-logits / self.temperature))).astype(np.float32)


class IsotonicCalibrator:
    def __init__(self): self.iso = IsotonicRegression(out_of_bounds='clip')

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'IsotonicCalibrator':
        scores = 1.0 / (1.0 + np.exp(-logits))
        self.iso.fit(scores, labels)
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        scores = 1.0 / (1.0 + np.exp(-logits))
        return self.iso.predict(scores).astype(np.float32)


def get_calibrator():
    if CFG.CALIB_METHOD == 'temperature': return TemperatureScaler()
    if CFG.CALIB_METHOD == 'isotonic':   return IsotonicCalibrator()
    raise ValueError(f'Unknown calibration method: {CFG.CALIB_METHOD}')


def plot_calibration_curve(scores, labels, title='Calibration'):
    prob_true, prob_pred = calibration_curve(labels, scores, n_bins=10)
    plt.figure(figsize=(5, 5))
    plt.plot(prob_pred, prob_true, 'b-o', label='Model')
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
    plt.xlabel('Predicted probability')
    plt.ylabel('Fraction positives')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


print('TemperatureScaler, IsotonicCalibrator, get_calibrator defined.')

## 11. Inference and Submission

In [ ]:
TTA_TRANSFORMS = [
    build_val_transform(),
    A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=1.0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]),
]


@torch.no_grad()
def predict(
    model: nn.Module, df: pd.DataFrame,
    calibrator=None, use_tta: bool = True
) -> np.ndarray:
    """
    Run inference and return calibrated scores in [0, 1].
    Averages logits across TTA transforms before calibrating.
    Data: /kaggle/input/datasets/maheshwarmishra/freuid-data/
    """
    model.eval()
    transforms = TTA_TRANSFORMS if use_tta else [build_val_transform()]
    all_logits = []

    for tfm in transforms:
        ds = FREUIDDataset(df, data_dir, tfm, is_train=False)
        loader = DataLoader(
            ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
        tfm_logits = []
        for batch in loader:
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                out = model(batch['image'].to(DEVICE),
                            batch['is_digital'].to(DEVICE),
                            batch['type_idx'].to(DEVICE))
            tfm_logits.append(out.cpu().float().numpy())
        if not tfm_logits:
            raise ValueError(
                f'DataLoader produced 0 batches. '
                f'df has {len(df)} rows — check that images exist.')
        all_logits.append(np.concatenate(tfm_logits))

    mean_logits = np.mean(all_logits, axis=0)
    if calibrator is not None:
        return calibrator.transform(mean_logits)
    return (1.0 / (1.0 + np.exp(-mean_logits))).astype(np.float32)


def generate_submission(
    scores: np.ndarray, test_df_: pd.DataFrame,
    filename: str = 'submission.csv', fill_value: float = 0.5
) -> pd.DataFrame:
    """
    Generate full 142818-row submission using sample_submission.csv as template.
    Source: /kaggle/input/datasets/maheshwarmishra/freuid-data/sample_submission.csv

    Real scores for 7821 public test images we predicted on.
    fill_value (training fraud rate) for 134997 private test IDs not yet released.
    Competition host confirmed dummy scores are ignored on public leaderboard.
    Score column name is 'label' (NOT 'score') in sample_submission.csv.
    """
    assert len(scores) == len(test_df_), (
        f'Length mismatch: {len(scores)} scores vs {len(test_df_)} rows')
    assert np.all((scores >= 0) & (scores <= 1)), 'Scores must be in [0,1]'
    if sub_df.empty:
        raise RuntimeError(
            'sample_submission.csv not found at '
            '/kaggle/input/datasets/maheshwarmishra/freuid-data/sample_submission.csv')

    score_col = [c for c in sub_df.columns if c != 'id']
    if not score_col:
        raise ValueError(f'No score column found in sample_submission: {sub_df.columns.tolist()}')
    score_col = score_col[0]
    print(f'Submission column: "{score_col}"')

    score_map         = dict(zip(test_df_['id'].astype(str), scores))
    full_sub          = sub_df.copy()
    full_sub[score_col] = full_sub['id'].astype(str).map(score_map).fillna(fill_value)

    n_real    = full_sub['id'].astype(str).isin(score_map).sum()
    n_default = len(full_sub) - n_real

    out_path = Path(CFG.OUTPUT_DIR) / filename
    full_sub.to_csv(out_path, index=False)
    print(f'Saved: {out_path} | rows={len(full_sub)} | '
          f'real={n_real} | fill({fill_value:.3f})={n_default}')
    print(f'Scores: min={scores.min():.4f} max={scores.max():.4f} '
          f'mean={scores.mean():.4f}')
    return full_sub


print('predict() and generate_submission() defined.')

## 12. Training Pipeline

In [ ]:
def build_scheduler(optimizer, n_epochs: int, steps_per_epoch: int):
    """Linear warmup then cosine annealing, operating on steps."""
    total_steps  = n_epochs * steps_per_epoch
    warmup_steps = CFG.WARMUP_EPOCHS * steps_per_epoch

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return (CFG.LR_MIN / CFG.LR +
                    (1.0 - CFG.LR_MIN / CFG.LR) *
                    step / max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return (CFG.LR_MIN / CFG.LR +
                0.5 * (1.0 - CFG.LR_MIN / CFG.LR) *
                (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(
    model, loader, optimizer, scheduler, loss_fn, scaler, epoch
) -> Dict[str, float]:
    """
    Single epoch with AMP, gradient accumulation, gradient clipping.
    FREUID NOT computed on training set (O(n^2) threshold sweeps would hang).
    Training monitored with loss + AUC only.
    """
    model.train()
    total_loss = 0.0
    all_logits, all_labels = [], []
    n_batches  = len(loader)
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=CFG.AMP and scaler is not None):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())
            loss = loss / CFG.GRAD_ACCUM

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0 or (step + 1) == n_batches:
            if scaler is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        all_logits.append(logits.detach().cpu().float())
        all_labels.append(lbl.detach().cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))
    try:
        trn_auc = roc_auc_score(labels_np, scores_np)
    except Exception:
        trn_auc = float('nan')
    return {'loss': total_loss / n_batches, 'auc_roc': trn_auc,
            'freuid': float('nan'), 'audet': float('nan'),
            'apcer_1pct': float('nan')}


@torch.no_grad()
def validate(
    model, loader, loss_fn
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    """Full validation with FREUID Score computation."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    for batch in loader:
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG.AMP):
            logits = model(img, dig, tidx)
            if isinstance(loss_fn, CombinedLoss):
                loss, _ = loss_fn(logits, lbl)
            else:
                loss = loss_fn(logits, lbl.float())
        total_loss += loss.item()
        all_logits.append(logits.cpu().float())
        all_labels.append(lbl.cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))
    metrics         = compute_freuid_score(labels_np, scores_np)
    metrics['loss'] = total_loss / len(loader)
    return metrics, scores_np, logits_np, labels_np


print('build_scheduler, train_one_epoch, validate defined.')

In [ ]:
def train_fold(fold: int = 0) -> Tuple:
    """
    Train one cross-validation fold.

    Features:
    - Resume from checkpoint: continues from last saved epoch if crashed.
    - Fast val per epoch: VAL_SAMPLE_SIZE samples for speed monitoring.
    - Full val once at end: all val samples for final accurate metrics.
    - Dual checkpoint save: checkpoints/ AND output root for persistence.
    - Full augmentation: build_train_transform() with recapture simulation.

    Data: /kaggle/input/datasets/maheshwarmishra/freuid-data/
    """
    print(f'\n{"="*60}')
    print(f' Training Fold {fold} / {n_folds_safe - 1}')
    print(f'{"="*60}')

    trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f'Train: {len(trn_df)} | Val: {len(val_df)}')

    trn_ds = FREUIDDataset(trn_df, data_dir, build_train_transform(), is_train=True)

    val_fast_df = val_df.sample(
        n=min(CFG.VAL_SAMPLE_SIZE, len(val_df)), random_state=SEED
    ).reset_index(drop=True)
    val_ds_fast = FREUIDDataset(val_fast_df, data_dir, build_val_transform(), is_train=True)
    val_ds_full = FREUIDDataset(val_df, data_dir, build_val_transform(), is_train=True)

    sampler = build_weighted_sampler(trn_df)
    trn_loader = DataLoader(
        trn_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY, drop_last=True)
    val_loader_fast = DataLoader(
        val_ds_fast, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
    val_loader_full = DataLoader(
        val_ds_full, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

    model   = FREUIDModel().to(DEVICE)
    loss_fn = get_loss_fn()

    param_groups = [
        {'params': model.backbone.parameters(),  'lr': CFG.LR * 0.1},
        {'params': model.head.parameters(),      'lr': CFG.LR},
    ]
    if CFG.USE_METADATA:
        param_groups.append(
            {'params': model.doc_embedding.parameters(), 'lr': CFG.LR})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = build_scheduler(optimizer, CFG.EPOCHS, len(trn_loader))
    scaler    = (torch.cuda.amp.GradScaler()
                 if CFG.AMP and torch.cuda.is_available() else None)

    ckpt_path    = Path(CFG.CHECKPOINT_DIR) / f'fold{fold}_best.pth'
    resume_epoch = 0
    best_freuid  = float('inf')
    best_state   = None
    history      = []

    if ckpt_path.exists():
        print(f'Resuming from checkpoint: {ckpt_path}')
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        resume_epoch = ckpt['epoch'] + 1
        best_freuid  = ckpt['val_metrics']['freuid']
        best_state   = ckpt
        print(f'  Resumed from epoch {resume_epoch-1}, best FREUID={best_freuid:.4f}')
    else:
        print('No existing checkpoint. Training from scratch.')

    for epoch in range(resume_epoch, CFG.EPOCHS):
        t0    = time.time()
        trn_m = train_one_epoch(
            model, trn_loader, optimizer, scheduler, loss_fn, scaler, epoch)
        val_m, val_scores, val_logits, val_labels = validate(
            model, val_loader_fast, loss_fn)
        elapsed = time.time() - t0

        history.append({
            'epoch': epoch, 'trn_loss': trn_m['loss'], 'trn_auc': trn_m['auc_roc'],
            'val_loss': val_m['loss'], 'val_freuid': val_m['freuid'],
            'val_audet': val_m['audet'], 'val_apcer_1pct': val_m['apcer_1pct'],
            'val_auc': val_m['auc_roc'],
            'lr': optimizer.param_groups[0]['lr'], 'elapsed_s': elapsed,
        })

        print(
            f'  Ep {epoch:02d} | '
            f'trn_loss={trn_m["loss"]:.4f} trn_auc={trn_m["auc_roc"]:.4f} | '
            f'val_freuid={val_m["freuid"]:.4f} '
            f'(audet={val_m["audet"]:.4f}, apcer@1%={val_m["apcer_1pct"]:.4f}) | '
            f'val_auc={val_m["auc_roc"]:.4f} | t={elapsed:.0f}s'
        )

        if val_m['freuid'] < best_freuid:
            best_freuid = val_m['freuid']
            best_state  = {
                'model_state': model.state_dict(),
                'val_scores':  val_scores, 'val_logits': val_logits,
                'val_labels':  val_labels, 'val_metrics': val_m,
                'epoch': epoch, 'fold': fold,
            }
            torch.save(best_state, ckpt_path)
            shutil.copy2(ckpt_path, f'{CFG.OUTPUT_DIR}/fold{fold}_best.pth')
            print(f'    New best FREUID={best_freuid:.4f} saved.')

        # Clear GPU cache after each epoch to prevent memory fragmentation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Final full validation
    print('\nRunning final full validation...')
    model.load_state_dict(best_state['model_state'])
    val_m_full, val_s_full, val_l_full, val_lbl_full = validate(
        model, val_loader_full, loss_fn)
    print(
        f'Full val FREUID={val_m_full["freuid"]:.4f} '
        f'(audet={val_m_full["audet"]:.4f}, '
        f'apcer@1%={val_m_full["apcer_1pct"]:.4f}) | '
        f'val_auc={val_m_full["auc_roc"]:.4f}'
    )
    best_state.update({
        'val_scores': val_s_full, 'val_logits': val_l_full,
        'val_labels': val_lbl_full, 'val_metrics': val_m_full,
    })
    torch.save(best_state, ckpt_path)
    shutil.copy2(ckpt_path, f'{CFG.OUTPUT_DIR}/fold{fold}_best.pth')
    print('Final checkpoint saved.')

    # Calibration
    calibrator = None
    if CFG.CALIBRATE:
        print('Fitting calibrator...')
        calibrator = get_calibrator()
        calibrator.fit(best_state['val_logits'], best_state['val_labels'])
        cal_scores = calibrator.transform(best_state['val_logits'])
        m_before   = compute_freuid_score(
            best_state['val_labels'], best_state['val_scores'])
        m_after    = compute_freuid_score(
            best_state['val_labels'], cal_scores)
        print(f'  FREUID before calibration: {m_before["freuid"]:.4f}')
        print(f'  FREUID after  calibration: {m_after["freuid"]:.4f}')

    plot_det_curve(
        best_state['val_labels'], best_state['val_scores'],
        title=f'Fold{fold}_DET_Curve')

    return model, calibrator, history, best_state


print('train_fold() defined.')

## 13. Diagnostics

In [ ]:
# ── Diagnostic 1: is_digital / type vs label leakage check ─────────────
print('=== label × is_digital ===')
ct1 = pd.crosstab(train_df['label'], train_df['is_digital'], margins=True)
print(ct1)
print('\nRow-normalized (fraction fraud within each is_digital group):')
print(pd.crosstab(train_df['is_digital'], train_df['label'], normalize='index'))

print('\n=== label × type ===')
ct2 = pd.crosstab(train_df['label'], train_df['type'], margins=True)
print(ct2)
print('\nRow-normalized (fraction fraud within each type):')
print(pd.crosstab(train_df['type'], train_df['label'], normalize='index'))

# Flag if any single is_digital value or type is >90% one class —
# that's a strong shortcut candidate for the model to exploit.
print('\n=== Leakage flags (>90% skew) ===')
dig_skew = pd.crosstab(train_df['is_digital'], train_df['label'], normalize='index')
for idx, row in dig_skew.iterrows():
    if row.max() > 0.90:
        print(f'  FLAG: is_digital={idx} is {row.max():.1%} single-class')

type_skew = pd.crosstab(train_df['type'], train_df['label'], normalize='index')
for idx, row in type_skew.iterrows():
    if row.max() > 0.90:
        print(f'  FLAG: type={idx} is {row.max():.1%} single-class')

In [ ]:
# ── Diagnostic 3: train/val near-duplicate check via perceptual hashing ─
# If val images have near-identical counterparts in train, "validation"
# is partly memorization, not generalization.
try:
    import imagehash
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'imagehash', '-q'])
    import imagehash

trn_df_fold0 = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)

def compute_hashes(df, n_sample=None, seed=SEED):
    d = df if n_sample is None else df.sample(n=min(n_sample, len(df)), random_state=seed)
    hashes = {}
    for _, row in d.iterrows():
        p = resolve_image_path(data_dir, row['image_path'])
        try:
            hashes[row['id']] = imagehash.phash(Image.open(p).convert('RGB'))
        except Exception:
            continue
    return hashes

print('Hashing sample of train and val images (this may take a minute)...')
trn_hashes = compute_hashes(trn_df_fold0, n_sample=5000)
val_hashes = compute_hashes(val_df_fold0, n_sample=500)   # smaller — this is O(n*m)

# Compare each val hash against all sampled train hashes; flag close matches
THRESH = 5   # hamming distance; <=5 is generally considered near-duplicate
near_dupes = []
trn_hash_list = list(trn_hashes.items())
for val_id, vh in val_hashes.items():
    best_dist = min(vh - th for _, th in trn_hash_list)
    if best_dist <= THRESH:
        near_dupes.append((val_id, best_dist))

print(f'\nChecked {len(val_hashes)} val images against {len(trn_hashes)} train images.')
print(f'Near-duplicates found (hamming <= {THRESH}): {len(near_dupes)} '
      f'({len(near_dupes)/max(1,len(val_hashes)):.1%} of sampled val)')
if near_dupes[:10]:
    print('Sample matches:', near_dupes[:10])

In [ ]:
# ── Diagnostic 4: is this still the pre-June sample, or the full release? ─
print(f'Train rows       : {len(train_df)}')
print(f'Unique doc types : {train_df["type"].nunique()}')
print(f'Types present    : {sorted(train_df["type"].unique())}')
print(f'Test rows        : {len(test_df)}')
print(f'Sample submission rows: {len(sub_df)}')

In [ ]:
# ── Diagnostic 2: does the model rely on type_idx / is_digital? ────────
# Load fold 0's best checkpoint and re-run validation with metadata
# zeroed out. If FREUID stays low, the model isn't leaning on metadata.
# If FREUID jumps sharply, metadata over-reliance is confirmed.

ckpt = torch.load('/kaggle/input/datasets/maheshwarmishra/fold01/fold0_best.pth',
                   map_location=DEVICE, weights_only=False)
diag_model = FREUIDModel().to(DEVICE)
diag_model.load_state_dict(ckpt['model_state'])
diag_model.eval()

val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)
val_ds = FREUIDDataset(val_df_fold0, data_dir, build_val_transform(), is_train=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

@torch.no_grad()
def run_with_metadata_override(model, loader, zero_type=False, zero_digital=False):
    all_logits, all_labels = [], []
    for batch in loader:
        img  = batch['image'].to(DEVICE)
        dig  = batch['is_digital'].to(DEVICE)
        tidx = batch['type_idx'].to(DEVICE)
        lbl  = batch['label'].to(DEVICE)
        if zero_digital:
            dig = torch.zeros_like(dig)
        if zero_type:
            tidx = torch.zeros_like(tidx)   # 0 = <UNK>
        with torch.cuda.amp.autocast(enabled=CFG.AMP):
            logits = model(img, dig, tidx)
        all_logits.append(logits.cpu().float())
        all_labels.append(lbl.cpu())
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    scores_np = 1.0 / (1.0 + np.exp(-logits_np))
    return compute_freuid_score(labels_np, scores_np), scores_np

print('=== Metadata ablation on fold 0 checkpoint ===')
m_normal, _ = run_with_metadata_override(diag_model, val_loader)
print(f'Normal (real metadata)      : FREUID={m_normal["freuid"]:.4f} AUC={m_normal["auc_roc"]:.4f}')

m_no_type, _ = run_with_metadata_override(diag_model, val_loader, zero_type=True)
print(f'type_idx zeroed (<UNK>)     : FREUID={m_no_type["freuid"]:.4f} AUC={m_no_type["auc_roc"]:.4f}')

m_no_dig, _ = run_with_metadata_override(diag_model, val_loader, zero_digital=True)
print(f'is_digital zeroed           : FREUID={m_no_dig["freuid"]:.4f} AUC={m_no_dig["auc_roc"]:.4f}')

m_no_both, _ = run_with_metadata_override(diag_model, val_loader, zero_type=True, zero_digital=True)
print(f'Both zeroed (test-time sim) : FREUID={m_no_both["freuid"]:.4f} AUC={m_no_both["auc_roc"]:.4f}')

del diag_model
torch.cuda.empty_cache()

In [ ]:
# ── Diagnostic 6: Per-document-type performance breakdown ──────────────
# Uses the same ckpt loaded in Diagnostic 2. If some types score
# near-perfectly while others don't, that's a memorization signal
# distinct from (but related to) the metadata-reliance check above.

# Safety check: make sure ckpt is loaded (in case this cell runs standalone)
if 'ckpt' not in dir() or ckpt is None:
    ckpt = torch.load('/kaggle/input/datasets/maheshwarmishra/fold01/fold0_best.pth',
                       map_location=DEVICE, weights_only=False)

breakdown = evaluate_breakdown(val_df_fold0, ckpt['val_scores'])

print('=== Performance by Document Type / is_digital ===')
print(breakdown[['group', 'subset', 'n', 'freuid', 'audet', 'apcer_1pct', 'auc_roc']]
      .sort_values('freuid').to_string(index=False))

# Flag near-perfect subsets — a red flag if it's isolated to specific
# types/groups rather than uniformly strong across all of them
perfect = breakdown[breakdown['freuid'] < 0.001]
if len(perfect) > 0:
    print(f'\n⚠️  {len(perfect)} group(s) have near-perfect FREUID (<0.001):')
    print(perfect[['group', 'subset', 'n', 'freuid']].to_string(index=False))

# Flag any subset with n too small to be statistically meaningful —
# a "perfect" score on 20-30 samples means much less than on thousands
small_n = breakdown[breakdown['n'] < 50]
if len(small_n) > 0:
    print(f'\nNote: {len(small_n)} group(s) have n<50 — treat their scores with caution:')
    print(small_n[['group', 'subset', 'n', 'freuid']].to_string(index=False))

In [ ]:
# ── Diagnostic 8: Visual side-by-side of "near-duplicate" matched pairs ──
# Diagnostic 3 found 97.8% of val images had a train match at hamming<=5.
# This could mean genuine duplication, OR it could mean phash just can't
# distinguish documents that share the same template/border/layout.
# Look at actual pairs to tell which one it is.

import matplotlib.pyplot as plt

# Rebuild a small set of matches with full (val_id, train_id, distance) info
# so we can actually open and compare the images, not just count them.
trn_df_fold0 = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)

trn_sample = trn_df_fold0.sample(n=min(2000, len(trn_df_fold0)), random_state=SEED)
val_sample = val_df_fold0.sample(n=20, random_state=SEED)   # small — just for eyeballing

trn_hashes = {}
for _, row in trn_sample.iterrows():
    p = resolve_image_path(data_dir, row['image_path'])
    try:
        trn_hashes[row['id']] = (imagehash.phash(Image.open(p).convert('RGB')), row)
    except Exception:
        continue

pairs_to_show = []
for _, vrow in val_sample.iterrows():
    p = resolve_image_path(data_dir, vrow['image_path'])
    try:
        vh = imagehash.phash(Image.open(p).convert('RGB'))
    except Exception:
        continue
    best_dist, best_tid, best_trow = None, None, None
    for tid, (th, trow) in trn_hashes.items():
        d = vh - th
        if best_dist is None or d < best_dist:
            best_dist, best_tid, best_trow = d, tid, trow
    pairs_to_show.append((vrow, best_trow, best_dist))

pairs_to_show.sort(key=lambda x: x[2])   # closest matches first

n_show = min(6, len(pairs_to_show))
fig, axes = plt.subplots(n_show, 2, figsize=(8, 4 * n_show))
for i in range(n_show):
    vrow, trow, dist = pairs_to_show[i]
    v_path = resolve_image_path(data_dir, vrow['image_path'])
    t_path = resolve_image_path(data_dir, trow['image_path'])
    try:
        axes[i, 0].imshow(Image.open(v_path).convert('RGB'))
    except Exception:
        pass
    axes[i, 0].set_title(f"VAL id={vrow['id'][:8]} | label={vrow['label']} | {vrow['type']}", fontsize=9)
    axes[i, 0].axis('off')
    try:
        axes[i, 1].imshow(Image.open(t_path).convert('RGB'))
    except Exception:
        pass
    axes[i, 1].set_title(f"TRAIN id={trow['id'][:8]} | label={trow['label']} | dist={dist}", fontsize=9)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/dup_check_visual.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nShowing {n_show} closest val→train matches, sorted by hamming distance.')
print('Look for: identical documents (same person/data, maybe different crop/compression)')
print('  vs. merely same-template documents (different person, same layout/border/holograms).')

In [ ]:
# ── Diagnostic 9: Low-level pipeline fingerprint check (label=0 vs label=1) ─
# If genuine and fraud images were produced/post-processed by even slightly
# different pipelines, they'll differ in file-level statistics that have
# nothing to do with document content — compression, resolution, resave
# artifacts, high-frequency energy. A CNN will find and exploit these
# far more easily than real forensic cues.

import io
from PIL import Image, ExifTags
import numpy as np

def jpeg_quality_estimate(path):
    """Rough JPEG quality estimate via libjpeg quantization tables (0-100)."""
    try:
        img = Image.open(path)
        if 'jpeg' not in img.format.lower() if img.format else True:
            return None
        # PIL exposes quantization tables via img.quantization (dict of lists)
        qtables = getattr(img, 'quantization', None)
        if not qtables:
            return None
        # Standard heuristic: average of luma qtable[0], lower avg = higher quality
        avg = np.mean(qtables[0])
        # Rough inverse mapping (not exact, but consistent/comparable across images)
        return max(1, min(100, 100 - avg))
    except Exception:
        return None

def high_freq_energy(path, size=256):
    """Fraction of FFT energy in the high-frequency band — proxy for
    recompression/resampling artifacts or generator-specific texture."""
    try:
        img = np.array(Image.open(path).convert('L').resize((size, size)), dtype=np.float32)
        f = np.fft.fftshift(np.fft.fft2(img))
        mag = np.abs(f)
        h, w = mag.shape
        cy, cx = h // 2, w // 2
        low_mask = np.zeros_like(mag, dtype=bool)
        r = min(h, w) // 8   # low-frequency radius
        yy, xx = np.ogrid[:h, :w]
        low_mask[((yy - cy)**2 + (xx - cx)**2) <= r**2] = True
        total = mag.sum() + 1e-8
        high_energy = mag[~low_mask].sum()
        return float(high_energy / total)
    except Exception:
        return None

def collect_stats(df, n_per_class=300, seed=SEED):
    records = []
    for lbl in [0, 1]:
        sub = df[df['label'] == lbl].sample(
            n=min(n_per_class, (df['label'] == lbl).sum()), random_state=seed)
        for _, row in sub.iterrows():
            p = resolve_image_path(data_dir, row['image_path'])
            try:
                file_size = p.stat().st_size
                img = Image.open(p)
                w, h = img.size
                has_exif = bool(img.getexif())
                fmt = img.format
            except Exception:
                continue
            q = jpeg_quality_estimate(p)
            hf = high_freq_energy(p)
            records.append({
                'id': row['id'], 'label': lbl, 'type': row['type'],
                'file_size': file_size, 'width': w, 'height': h,
                'format': fmt, 'has_exif': has_exif,
                'jpeg_quality_est': q, 'high_freq_energy': hf,
            })
    return pd.DataFrame(records)

print('Collecting low-level stats (this may take a couple minutes)...')
stats_df = collect_stats(train_df, n_per_class=300)
print(f'Collected stats for {len(stats_df)} images.\n')

# ── Summary by label ─────────────────────────────────────────────────
print('=== File size (bytes) by label ===')
print(stats_df.groupby('label')['file_size'].describe()[['mean', 'std', 'min', 'max']])

print('\n=== Resolution by label ===')
print(stats_df.groupby('label')[['width', 'height']].describe())

print('\n=== Format / EXIF presence by label ===')
print(pd.crosstab(stats_df['label'], stats_df['format']))
print(pd.crosstab(stats_df['label'], stats_df['has_exif']))

print('\n=== JPEG quality estimate by label ===')
q_valid = stats_df.dropna(subset=['jpeg_quality_est'])
if len(q_valid) > 0:
    print(q_valid.groupby('label')['jpeg_quality_est'].describe()[['mean', 'std', 'min', 'max']])
else:
    print('No JPEG quantization tables found (images may not be JPEG, or PIL stripped them).')

print('\n=== High-frequency FFT energy by label ===')
hf_valid = stats_df.dropna(subset=['high_freq_energy'])
print(hf_valid.groupby('label')['high_freq_energy'].describe()[['mean', 'std', 'min', 'max']])

# ── Quick visual check: histogram overlap ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for lbl, color in [(0, 'blue'), (1, 'red')]:
    sub = stats_df[stats_df['label'] == lbl]
    axes[0].hist(sub['file_size'], bins=30, alpha=0.5, label=f'label={lbl}', color=color)
    if q_valid is not None and len(q_valid) > 0:
        axes[1].hist(q_valid[q_valid['label'] == lbl]['jpeg_quality_est'],
                      bins=30, alpha=0.5, label=f'label={lbl}', color=color)
    axes[2].hist(hf_valid[hf_valid['label'] == lbl]['high_freq_energy'],
                  bins=30, alpha=0.5, label=f'label={lbl}', color=color)

axes[0].set_title('File size'); axes[0].legend()
axes[1].set_title('JPEG quality estimate'); axes[1].legend()
axes[2].set_title('High-freq FFT energy'); axes[2].legend()
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/pipeline_fingerprint_check.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n=== Interpretation guide ===')
print('If any histogram above shows label=0 and label=1 as cleanly SEPARATED')
print('(little/no overlap), that is a strong pipeline-fingerprint signal —')
print('the model may be learning file-level artifacts, not document forensics.')
print('If histograms overlap heavily, this particular check does not explain')
print('the near-perfect scores, and the fingerprint (if any) is more subtle.')

In [ ]:
# ── Diagnostic 10: Grad-CAM — where is the model actually looking? ──────
# Visualizes which image regions drive the fraud/genuine decision.
# Localized attention (e.g. always the face-photo box, or a specific
# field) suggests a splice/edit-boundary shortcut — which may or may not
# transfer to unseen document types in the private test set.

import torch.nn.functional as F
import matplotlib.cm as cm

ckpt = torch.load('/kaggle/input/datasets/maheshwarmishra/fold01/fold0_best.pth',
                   map_location=DEVICE, weights_only=False)   # reuse same path as Diagnostic 2/6
gc_model = FREUIDModel().to(DEVICE)
gc_model.load_state_dict(ckpt['model_state'])
gc_model.eval()

# ── Hook the last conv feature map (before global pooling) ─────────────
activations = {}
gradients = {}

def fwd_hook(module, inp, out):
    activations['feat'] = out.detach()

def bwd_hook(module, grad_in, grad_out):
    gradients['feat'] = grad_out[0].detach()

# tf_efficientnet_b4's final conv layer before global pool is conv_head
target_layer = gc_model.backbone.conv_head
h1 = target_layer.register_forward_hook(fwd_hook)
h2 = target_layer.register_full_backward_hook(bwd_hook)

def compute_gradcam(model, img_tensor, is_digital_val, type_idx_val):
    """img_tensor: (1, C, H, W) already normalized, on DEVICE, requires_grad not needed."""
    model.zero_grad()
    img_tensor = img_tensor.clone().requires_grad_(False)
    dig  = torch.tensor([is_digital_val], dtype=torch.float32, device=DEVICE)
    tidx = torch.tensor([type_idx_val], dtype=torch.long, device=DEVICE)

    logit = model(img_tensor, dig, tidx)   # (1,)
    logit.backward()

    acts = activations['feat'][0]      # (C, h, w)
    grads = gradients['feat'][0]       # (C, h, w)
    weights = grads.mean(dim=(1, 2))   # (C,) — global-avg-pooled gradients

    cam = torch.relu((weights[:, None, None] * acts).sum(dim=0))  # (h, w)
    cam = cam / (cam.max() + 1e-8)
    return cam.cpu().numpy(), torch.sigmoid(logit).item()

# ── Pick a handful of images across both labels ─────────────────────────
val_df_fold0 = train_df[train_df['fold'] == 0].reset_index(drop=True)
gc_sample = pd.concat([
    val_df_fold0[val_df_fold0['label'] == 0].sample(4, random_state=SEED),
    val_df_fold0[val_df_fold0['label'] == 1].sample(4, random_state=SEED),
])

val_tfm = build_val_transform()
n = len(gc_sample)
fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))

for i, (_, row) in enumerate(gc_sample.iterrows()):
    p = resolve_image_path(data_dir, row['image_path'])
    raw_img = Image.open(p).convert('RGB')
    raw_np = np.array(raw_img.resize((CFG.IMG_SIZE, CFG.IMG_SIZE)))

    transformed = val_tfm(image=np.array(raw_img))['image']
    img_tensor = transformed.unsqueeze(0).to(DEVICE)

    cam, pred_score = compute_gradcam(
        gc_model, img_tensor,
        is_digital_val=row.get('is_digital', 0),
        type_idx_val=row.get('type_idx', 0),
    )
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8))
                            .resize((CFG.IMG_SIZE, CFG.IMG_SIZE), Image.BILINEAR)) / 255.0

    axes[i, 0].imshow(raw_np)
    axes[i, 0].set_title(f"label={row['label']} | {row['type']} | pred={pred_score:.3f}", fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(raw_np)
    axes[i, 1].imshow(cam_resized, cmap='jet', alpha=0.5)
    axes[i, 1].set_title('Grad-CAM overlay', fontsize=9)
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/gradcam_check.png', dpi=100, bbox_inches='tight')
plt.show()

h1.remove()
h2.remove()
del gc_model
torch.cuda.empty_cache()

print('Look for: does the hot region consistently land on the SAME area')
print('(e.g. always the photo box, always a corner, always the border/hologram)')
print('regardless of document type or label? That points to a localized shortcut.')
print('Diffuse/whole-document attention that varies by image content is more')
print('consistent with genuine (if possibly still non-transferable) forensic learning.')